# 行動パターンからメタパラメータ、価値関数の推定するプログラム
## 1. モジュールの呼び出し

In [3]:
# numpyモジュールの呼び出し
import numpy as np

# matplotlibモジュールの呼び出し (%matplotlib行は環境に応じて変更すること)
%matplotlib osx
# %matplotlib inline
# %matplotlib tk
import matplotlib.pyplot as plt

# seabornモジュールの呼び出し（heatmapのため）
import seaborn as sns

## 2. 環境クラスを生成
### 2.1 刺激依存二者択一課題（環境変化なし）のクラスとして定義

In [4]:
class SimpleChoiceTask():
    """
    刺激依存二者択一課題を定義するクラス
    [メンバー変数]
    - num_state: 状態の総数
    - num_action: 行動の総数
    - state: 各時刻で訪問している状態のインデクス
      - state = 0 -> 刺激パターン1
      - state = 1 -> 刺激パターン2
      にそれぞれ対応
    """
    # 状態の総数を表す定数
    num_state = 2
    # 行動の総数を表す定数
    num_action = 2
    # 現在の状態を表す変数
    state = 0
    
    def __init__(self):
        """
        コンストラクタ
        """
        pass
    
    def show_state(self):
        """
        現在の状態を標準出力するメソッド関数
        """
        print(self.state)

    def get_env_config(self):
        """
        環境設定（状態数, 行動数）を取得するためのメソッド関数
        """
        return self.num_state, self.num_action
    
    def init_state(self):
        """
        状態を初期化（初期状態に戻す）メソッド関数
        """
        rnd = np.random.choice(self.num_action, 1, 0.5)
        if rnd <= 0.5:
            self.state = 0
        else:
            self.state = 1
        
        # 報酬は常にnull
        reward = None
        return reward, self.state
        
    # 選択した行動に従って状態を遷移させるメソッド関数
    # state = 0の時:  action = 0のみ報酬がもらえる
    # state = 1の時: action = 1のみ報酬がもらえる
    def move_state(self, action):
        """
        選択した行動 action に従って状態を遷移させるメソッド関数
        [入力引数]
        - action: 行動のインデクス
          - action = 0: 左
          - action = 1: 右
        """
        if self.state == 0:
            if action == 0:
                reward = 1.0
            else:
                reward = 0.0
        else:
            if action == 0:
                reward = 0.0
            else:
                reward = 1.0
        
        # ゴール状態（2）へ常に移動
        self.state = 2
        return reward, self.state
    
    def reach_goal(self):
        """
        ゴールに到達したか否かを伝えるメソッド関数
        [返り値]
            - True: ゴールに到達
            - False: ゴールに未到達
        """
        if self.state >= self.num_state:
            return True
        else:
            return False

### 2.2 刺激依存二者択一課題（環境変化あり）のクラスとして定義

In [5]:
class ComplexChoiceTask():
    """
    刺激依存二者択一課題を定義するクラス
    [メンバー変数]
    - num_state: 状態の総数
    - num_action: 行動の総数
    - state: 各時刻で訪問している状態のインデクス
      - state = 0 -> 刺激パターン1
      - state = 1 -> 刺激パターン2
      にそれぞれ対応
    """
    # 状態の総数を表す定数
    num_state = 2
    # 行動の総数を表す定数
    num_action = 2
    # 現在の状態を表す変数
    state = 0
    # 課題成功率
    suc_rate = 0.0
    # 環境タイプ
    env_type = 0
    
    def __init__(self):
        """
        コンストラクタ
        """
        pass
    
    def show_state(self):
        """
        現在の状態を標準出力するメソッド関数
        """
        print(self.state)

    def get_env_config(self):
        """
        環境設定（状態数, 行動数）を取得するためのメソッド関数
        """
        return self.num_state, self.num_action
    
    def init_state(self):
        """
        状態を初期化（初期状態に戻す）メソッド関数
        """
        rnd = np.random.rand()
        if rnd <= 0.5:
            self.state = 0
        else:
            self.state = 1
        
        if self.suc_rate >= 0.8:
            self.suc_rate = 0.0
            if self.env_type == 0:
                self.env_type = 1
            else:
                self.env_type = 0        
        
        # 報酬は常にnull
        reward = None
        return reward, self.state
        
    # 選択した行動に従って状態を遷移させるメソッド関数
    # state = 0の時:  action = 0のみ報酬がもらえる
    # state = 1の時: action = 1のみ報酬がもらえる
    def move_state(self, action):
        """
        選択した行動 action に従って状態を遷移させるメソッド関数
        [入力引数]
        - action: 行動のインデクス
          - action = 0: 左
          - action = 1: 右
        """
        if self.state == 0:
            if action == 0:
                if self.env_type == 0:
                    reward = 1.0
                else:
                    reward = 0.0
            else:
                if self.env_type == 0:
                    reward = 0.0
                else:
                    reward = 1.0
        else:
            if action == 0:
                if self.env_type == 0:
                    reward = 0.0
                else:
                    reward = 1.0
            else:
                if self.env_type == 0:
                    reward = 1.0
                else:
                    reward = 0.0
        self.suc_rate = 0.9 * self.suc_rate + 0.1 * reward
        # ゴール状態（2）へ常に移動
        self.state = 2
        return reward, self.state
    
    def reach_goal(self):
        """
        ゴールに到達したか否かを伝えるメソッド関数
        [返り値]
            - True: ゴールに到達
            - False: ゴールに未到達
        """
        if self.state >= self.num_state:
            return True
        else:
            return False

## 3. Q学習エージェントクラスを作成 

In [6]:
class Qagent():
    """
    Q学習エージェントを定義するクラス
    """
    
    def __init__(self, num_state=1, num_action=1):
        """
        コンストラクタ
        [入力引数]
        - num_state: 環境の状態数 (default = 1)
        - num_action: 選択可能な行動数 (default = 1)
        """
        self.num_state = num_state
        self.num_action = num_action
        self.Q = np.zeros((num_state, num_action)) 
        return

    def set_meta_parameter(self, alpha = 0.1, beta = 1, gamma = 0.99):
        """
        メタパラメータ（エージェントの個性）を設定するメソッド関数
        """
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        return

    def reset_memory(self):
        """
        過去の状態・行動の履歴をリセットする
        """
        self.s0 = None
        self.a0 = None
        return
    
    def select_action(self, state):
        """
        与えられた状態に対して行動を選択するメソッド関数（方策関数）
        """
        # 行動選択に必要な変数の取得
        Q = self.Q
        beta = self.beta
        s = state
        
        # \pi(s,a) = \Pr(a|s) ~ exp[beta * Q(s,a)] を計算
        logp = self.beta * self.Q[state,:]
        max_logp = np.max(logp)
        un_p = np.exp(logp - max_logp)
        prob = un_p/sum(un_p)
        
        # \Pr(a|s)にしたがってaction aをサンプリング
        action = np.random.choice(self.num_action, 1, p=prob)
        a = action[0]
        return a
    
    def update_value(self, rwd, state, action):
        """
        与えられた状態に対して行動を選択するメソッド関数（方策関数）
        """
        # 学習や行動選択に必要な変数の取得
        Q = self.Q
        s0 = self.s0
        a0 = self.a0
        r1 = rwd
        s1 = state
        a1 = action
        alpha = self.alpha
        gamma = self.gamma
        
        # 行動価値関数の学習（1ステップ前の状態が観測できているときのみ）
        if self.s0 != None:
            if s1 < self.num_state:
                max_q = np.max(Q[s1,:])
            else:
                max_q = 0
            delta = r1 + gamma * max_q - Q[s0,a0]
            Q[s0,a0] = Q[s0,a0] + alpha * delta
            self.Q = Q
            
        # 過去の状態・行動の履歴を更新
        self.s0 = s1
        self.a0 = a1
                
        return

    def print_action_value(self):
        """
        現在の行動価値関数を標準出力するメソッド関数
        """
        print(self.Q)
        return
    
    def draw_action_value(self):
        """
        現在の行動価値関数をヒートマップ表示するメソッド関数
        """
        plt.figure(2)
        plt.imshow(self.Q,interpolation='nearest',cmap='jet')
        plt.pause(0.05)
        plt.show()

## 4. 強化学習エージェントによる行動パターンの生成

In [22]:
# 最大トライアル数を設定
MaxTrials = 2000

# 1トライアルあたりの最大ステップ数（行動選択回数）を設定
MaxSteps = 100

# 状態-行動系列の出力先ファイル名を指定する
filename = 'action_log_human.txt'

# 状態-行動系列を保存する配列を初期化する
sar_log = np.zeros((MaxTrials, 3)) 

# 環境のオブジェクトを生成
env1 = SimpleChoiceTask()

# 環境設定を取得する
[num_state, num_action] = env1.get_env_config()

# 学習エージェントオブジェクトを生成
agent1 = Qagent(num_state, num_action)

# 学習エージェントのメタパラメータを設定
agent1.set_meta_parameter(alpha=0.1, beta=2.0, gamma=1.0)

# 各トライアルに獲得した累積報酬を格納する変数を初期化
cum_rwd = np.zeros(MaxTrials)
# トライアルをMaxTrials回くり返す
for trial in range(MaxTrials):
    # 過去の状態・行動履歴をリセットする
    agent1.reset_memory()
    # 環境を初期状態に戻し、その状態を取得する
    [rwd, state] = env1.init_state()
    # ゴール状態に到達するまで以下をくり返す
    for step in range(MaxSteps):
        # ゴール状態に到達できたら、価値関数の更新のみ行う
        if env1.reach_goal():
            agent1.update_value(rwd, state, action)
            break
        # エージェントに現在の状態を伝えて、行動を選択させる
        action = agent1.select_action(state)
        # !!! 現在の状態をログ変数に保存
        sar_log[trial,0] = state
        # !!! 現在の行動をログ変数に保存
        sar_log[trial,1] = action
        # エージェントの価値関数を更新
        agent1.update_value(rwd, state, action)
        # 選択した行動を環境に出力し、次の状態に遷移させる
        [rwd, state] = env1.move_state(action)
        # !!! 行動によって得られた報酬をログ変数に保存
        sar_log[trial,2] = rwd
        # 得られた報酬を累積する
        cum_rwd[trial] = cum_rwd[trial] + rwd
    """
    # トライアル終了後に累積報酬の推移をグラフ表示
    plt.figure(1)
    plt.clf()
    plt.plot(range(1,trial+1), cum_rwd[0:trial], label = 'raw')
    if trial > 20:
        cum_rwd_mvavg =  np.convolve(cum_rwd[0:trial], np.ones(10)/10, mode='same')
        plt.plot(range(1,trial+1), cum_rwd_mvavg, label = 'mov. avg. (10)')
    plt.xlabel('Number of Trials')
    plt.ylabel('Cumurative Reward')
    plt.legend()
    plt.pause(0.05)
    plt.show()
    """
# ログを保存
np.savetxt(filename, sar_log, fmt='%f')



## 5. メタパラメータ・フィッティングクラスの作成

In [20]:
class Qfit():
    """
    Q学習エージェントを定義するクラス
    """
    def __init__(self, sar_log, num_state, num_action):
        """
        コンストラクタ
        [入力引数]
        - sar_log: 状態-行動-報酬時系列
        - num_state: 選択可能な状態数
        - num_action: 選択可能な行動数
        """
        # 時系列を保存
        self.sar_log = sar_log
                
        # 状態数と行動数を取得
        self.num_state = num_state
        self.num_action = num_action
        
        # 総トライアル数を取得する
        self.MaxTrials = sar_log.shape[0]
        
        return
    
    def calc_log_likelihood(self, alpha, beta):
        """
        対数尤度を計算するメソッド関数
        [入力引数]
        - alpha: 学習率
        - beta: 逆温度
        """
        # 行動価値関数を初期化
        Q = np.zeros((self.num_state, self.num_action))
        # 対数尤度を初期化
        llh = 0
        for n in range(self.MaxTrials):
            # 各時点の状態-行動-報酬を取得
            state = int(self.sar_log[n,0])
            action = int(self.sar_log[n,1])
            rwd = self.sar_log[n,2]
            # 各時点の対数尤度を計算する
            logp = beta * Q[state,:]
            max_logp = np.max(logp)
            un_p = np.exp(logp -max_logp)
            nf = np.sum(un_p)
            llh = llh + (logp[action] -max_logp - np.log(nf))
            # 行動価値関数の更新
            td = rwd - Q[state,action]
            Q[state,action] = Q[state,action] + alpha * td
        
        return llh
    
    def estimate_value_function(self, alpha, beta):
        """
        行動価値関数を補間するメソッド関数
        [入力引数]
        - alpha: 学習率
        - beta: 逆温度
        """
        # 行動価値関数を初期化
        Q = np.zeros((self.num_state, self.num_action))
        # 行動価値関数ログのメモリ確保
        Q_log = np.zeros((self.MaxTrials,self.num_state, self.num_action))
        for n in range(self.MaxTrials):
            # 行動価値関数ログを保存
            Q_log[n,:,:] = Q
            # 各時点の状態-行動-報酬を取得
            state = int(self.sar_log[n,0])
            action = int(self.sar_log[n,1])
            rwd = self.sar_log[n,2]
            # 行動価値関数の更新
            td = rwd - Q[state,action]
            Q[state,action] = Q[state,action] + alpha * td
        
        return Q_log
    
    def estimate_policy(self, alpha, beta):
        """
        行動価値関数を補間するメソッド関数
        [入力引数]
        - alpha: 学習率
        - beta: 逆温度
        """
        # 行動価値関数を初期化
        Q = np.zeros((self.num_state, self.num_action))
        # 行動価値関数ログのメモリ確保
        policy_log = np.zeros((self.MaxTrials,self.num_state, self.num_action))
        for n in range(self.MaxTrials):
            # 各時点のpolicyを計算する
            for s in range(self.num_state):
                logp = beta * Q[s,:]
                max_logp = np.max(logp)
                un_p = np.exp(logp -max_logp)
                prob = un_p/np.sum(un_p)
                policy_log[n,s,:] = prob
            # 各時点の状態-行動-報酬を取得
            state = int(self.sar_log[n,0])
            action = int(self.sar_log[n,1])
            rwd = self.sar_log[n,2]
            # 行動価値関数の更新
            td = rwd - Q[state,action]
            Q[state,action] = Q[state,action] + alpha * td
        
        return policy_log
    
    def estimate_td_error(self, alpha, beta):
        """
        TD誤差を補間するメソッド関数
        [入力引数]
        - alpha: 学習率
        - beta: 逆温度
        """
        # 行動価値関数を初期化
        Q = np.zeros((self.num_state, self.num_action))
        # 行動価値関数ログのメモリ確保
        td_log = np.zeros(self.MaxTrials)
        for n in range(self.MaxTrials):
            # 各時点の状態-行動-報酬を取得
            state = int(self.sar_log[n,0])
            action = int(self.sar_log[n,1])
            rwd = self.sar_log[n,2]
            # 行動価値関数の更新
            td = rwd - Q[state,action]
            Q[state,action] = Q[state,action] + alpha * td
            # TDログを保存
            td_log[n] = td
        return td_log

## 6. 行動パターンからメタパラメータの逆推定

In [24]:
# 状態-行動系列の読み込みファイル名を指定する
filename = 'action_log_human.txt'

# メタパラメータの探索範囲を指定
alpha_min = 0 # alphaの最小値
alpha_max = 1 # alphaの最大値
beta_min = 0 # betaの最小値
beta_max = 5 # betaの最大値
num_grids = 11 # 各メタパラメータ座標のグリッドの数

# 状態-行動系列を配列に読み込む
sar_log = np.loadtxt(filename)

# 環境のオブジェクトを生成
env1 = SimpleChoiceTask()

# 環境設定を取得する
[num_state, num_action] = env1.get_env_config()

# モデルフィット用オブジェクトの作成
fit1 = Qfit(sar_log, num_state, num_action)

# 探索のためのグリッドを設定する
alpha_grid = np.linspace(alpha_min, alpha_max, num_grids)
beta_grid = np.linspace(beta_min, beta_max, num_grids)

# 対数尤度を保存するための配列を用意する
llh_array = np.zeros((num_grids, num_grids))

# 各グリッド点におけるモデルの適合度（対数尤度）を求める
for ii in range(num_grids):
    alpha = alpha_grid[ii]
    for jj in range(num_grids):
        beta = beta_grid[jj]
        llh_array[ii,jj] = fit1.calc_log_likelihood(alpha, beta)

# 探索結果をheatmap表示する
plt.figure(1)
xticklabel = np.round(beta_grid,2)
yticklabel = np.round(alpha_grid,2)
sns.heatmap(llh_array, xticklabels = xticklabel, yticklabels= yticklabel)
plt.show()

# 最適メタパラメータを表示する
idx =np.unravel_index(llh_array.argmax(), llh_array.shape)
alpha_best = alpha_grid[idx[0]]
beta_best = beta_grid[idx[1]]
print('Best value of "alpha" = %f\n' % alpha_best)
print('Best value of "beta" = %f\n' % beta_best)

Best value of "alpha" = 0.100000

Best value of "beta" = 2.000000



## 7. 最適メタパラメータによる内部状態推定
### 7.1 行動価値関数の推定

In [26]:
# 行動価値関数の時間変化を取得
Q_log = fit1.estimate_value_function(alpha_best, beta_best)
# 行動ログと行動価値関数をプロット
plt.figure(1)
MaxTrials = sar_log.shape[0]
for n in range(MaxTrials):
    state = sar_log[n,0]
    action = sar_log[n,1]
    rwd = sar_log[n,2]
    if np.abs(rwd) < 0.1:
        length = 0.1
    else:
        length = 0.3
    if state == 0:
        color = 'blue'
        if action == 0:
            plt.plot([n+1,n+1], [-0.1,-0.1-length], color = color)
        else:
            plt.plot([n+1,n+1], [1.1,1.1+length], color = color)
    else:
        color = 'green'
        if action == 0:
            plt.plot([n+1,n+1], [-0.1,-0.1-length], color = color)
        else:
            plt.plot([n+1,n+1], [1.1,1.1+length], color = color)

plt.plot(range(1,MaxTrials+1),Q_log[:,0,0], color = 'blue', linestyle = 'dotted')
plt.plot(range(1,MaxTrials+1),Q_log[:,0,1], color = 'blue', linestyle = 'solid')
plt.plot(range(1,MaxTrials+1),Q_log[:,1,0], color = 'green', linestyle = 'dotted')
plt.plot(range(1,MaxTrials+1),Q_log[:,1,1], color = 'green', linestyle = 'solid')
plt.title("Estimated Q-value")
plt.show()

### 7.2 方策（行動選択確率）の推定

In [27]:
# 行動価値関数の時間変化を取得
policy_log = fit1.estimate_policy(alpha_best, beta_best)
# 行動ログと行動価値関数をプロット
plt.figure(1)
MaxTrials = sar_log.shape[0]
for n in range(MaxTrials):
    state = sar_log[n,0]
    action = sar_log[n,1]
    rwd = sar_log[n,2]
    if np.abs(rwd) < 0.1:
        length = 0.1
    else:
        length = 0.3
    if state == 0:
        color = 'blue'
        if action == 0:
            plt.plot([n+1,n+1], [-0.1,-0.1-length], color = color)
        else:
            plt.plot([n+1,n+1], [1.1,1.1+length], color = color)
    else:
        color = 'green'
        if action == 0:
            plt.plot([n+1,n+1], [-0.1,-0.1-length], color = color)
        else:
            plt.plot([n+1,n+1], [1.1,1.1+length], color = color)

plt.plot(range(1,MaxTrials+1),policy_log[:,0,1], color = 'blue', linestyle = 'solid')
plt.plot(range(1,MaxTrials+1),policy_log[:,1,1], color = 'green', linestyle = 'solid')
plt.title("Estimated Policy")
plt.show()

### 7.3 TD誤差の推定

In [29]:
# 方策の時間変化を取得
td_log = fit1.estimate_td_error(alpha_best, beta_best)
# 行動ログと行動価値関数をプロット
plt.figure(1)
MaxTrials = sar_log.shape[0]
for n in range(MaxTrials):
    state = sar_log[n,0]
    action = sar_log[n,1]
    rwd = sar_log[n,2]
    if np.abs(rwd) < 0.1:
        length = 0.1
    else:
        length = 0.3
    if state == 0:
        color = 'blue'
        if action == 0:
            plt.plot([n+1,n+1], [-0.1,-0.1-length], color = color)
        else:
            plt.plot([n+1,n+1], [1.1,1.1+length], color = color)
    else:
        color = 'green'
        if action == 0:
            plt.plot([n+1,n+1], [-0.1,-0.1-length], color = color)
        else:
            plt.plot([n+1,n+1], [1.1,1.1+length], color = color)

plt.plot(range(1,MaxTrials+1),td_log, color = 'red', linestyle = 'dotted')
plt.title("Estimated TD error")
plt.show()

## 8. 二者択一課題のヒトの行動実験（行動ログ収集）

In [ ]:
# 最大トライアル数を設定
MaxTrials = 20

# 1トライアルあたりの最大ステップ数（行動選択回数）を設定
MaxSteps = 100

# 状態-行動系列の出力先ファイル名を指定する
filename = 'action_log_human.txt'

# 状態-行動系列を保存する配列を初期化する
sar_log = np.zeros((MaxTrials, 3)) 

# 環境のオブジェクトを生成
env1 = ComplexChoiceTask()

# 環境設定を取得する
[num_state, num_action] = env1.get_env_config()

# 各トライアルに獲得した累積報酬を格納する変数を初期化
cum_rwd = np.zeros(MaxTrials)
# トライアルをMaxTrials回くり返す
for trial in range(MaxTrials):
    # 環境を初期状態に戻し、その状態を取得する
    [rwd, state] = env1.init_state()
    # ゴール状態に到達するまで以下をくり返す
    for step in range(MaxSteps):
        # ゴール状態に到達できたら、価値関数の更新のみ行う
        if env1.reach_goal():
            break
        # 刺激を表示
        print('==== trial: %d ====' % (trial+1))
        print('current state = %d' % state)
        while True:
            print('select action (0 or 1):')
            str = input()
            if str.isdecimal():
                action = int(str)
                if action == 0:
                    break
                if action ==1:
                    break;
        # !!! 現在の状態をログ変数に保存
        sar_log[trial,0] = state
        # !!! 現在の行動をログ変数に保存
        sar_log[trial,1] = action
        # 選択した行動を環境に出力し、次の状態に遷移させる
        [rwd, state] = env1.move_state(action)
        print('reward = %f\n\n\n' % rwd)
        # !!! 行動によって得られた報酬をログ変数に保存
        sar_log[trial,2] = rwd
        # 得られた報酬を累積する
        cum_rwd[trial] = cum_rwd[trial] + rwd
    # トライアル終了後に累積報酬の推移をグラフ表示
    """
    plt.figure(1)
    plt.clf()
    plt.plot(range(1,trial+1), cum_rwd[0:trial], label = 'raw')
    if trial > 20:
        cum_rwd_mvavg =  np.convolve(cum_rwd[0:trial], np.ones(10)/10, mode='same')
        plt.plot(range(1,trial+1), cum_rwd_mvavg, label = 'mov. avg. (10)')
    plt.xlabel('Number of Trials')
    plt.ylabel('Cumurative Reward')
    plt.legend()
    plt.pause(0.05)
    plt.show()
    """
    # ログを保存
np.savetxt(filename, sar_log, fmt='%f')



==== trial: 1 ====
current state = 0
select action (0 or 1):
0
reward = 1.000000



==== trial: 2 ====
current state = 1
select action (0 or 1):
1
reward = 1.000000



==== trial: 3 ====
current state = 1
select action (0 or 1):


## 9. 自由課題: 上記のスクリプトを参考に各自のメタパラメータを推定しよう!